# Notebook 21 — τ_ref availability audit (8 axis-(a)-runnable sets)

## Goal

Run a τ_ref availability audit on the 8 axis-(a)-runnable PyBaMM 26.3.1 DFN
parameter sets identified in Day 16 (notebook 20). The audit point corresponds to
`set_initial_state(0.05)` followed by an additional `Q_low = 0.05·Q_nom` charge
increment, nominally around SoC ≈ 10% within each parameter set and aligned with
the Cell 3A analysis-window entrance from Day 16.

The goal is **not** to rerun DC–AC immediately, but to determine whether τ_ref
varies enough across parameter families to make frequency rebasing informative.
DC–AC reruns are conditional on the τ_ref distribution observed here.

---

## Input parameter sets (8 axis-(a)-runnable, 5 chemistry-family groups)

| set | chemistry-family group |
|-----|------------------------|
| Ai2020 | LCO/graphite (Enertech) |
| Marquis2019 | LCO/graphite (Kokam-Marquis) |
| Chen2020 | LG M50 NMC811-Si lineage |
| OKane2022 | LG M50 NMC811-Si lineage |
| ORegan2022 | LG M50 NMC811-Si lineage |
| Mohtat2020 | graphite/NMC532 |
| NCA_Kim2011 | graphite/NCA |
| Ecker2015 | Kokam-Ecker Co-rich Mn-free layered oxide |

LFP, composite-electrode, and non-layered chemistries remain separate tasks
and are not part of this notebook.

---

## Protocol (frozen)

    set_initial_state(0.05)
    → charge at 0.2C for 900 s         (precharge to analysis-window entrance)
    → rest 600 s                       (pre-rest)
    → charge pulse at 1C for 10 s      (HPPC-style pulse)
    → rest 600 s                       (relaxation)

`Q_low = 0.05·Q_nom` is realized as `0.2C × 900 s = 0.05·Q_nom`, matching the
Cell 3A Day 16 reference path so that the τ_ref state point is consistent
with the DCAC analysis-window entrance.

PyBaMM execution path: `pybamm.Experiment` with multi-step protocol, sampling
period 5 ms on pulse + first 5 s of relaxation (for 20 ms CDEFG point
extraction), 1 s on the remainder of relaxation, 5 s on precharge / pre-rest.

Solver convention from Day 16 lock: `model.events` kept (not disabled);
PyBaMM convention I > 0 = discharge → `I_precharge = -0.2·Q_nom`,
`I_pulse = -1.0·Q_nom`. No callable current function is needed
(constant-current steps), so the `pybamm.sin` quirk does not apply here.

---

## Descriptor layer

**Primary candidate**: `tau2_biexp` — slow component of bi-exponential fit
to relaxation t ∈ [0.5 s, 600 s] starting from 0.5 s after pulse end (skipping
ohmic / fast-numerical transient). All four parameters (V_inf, A1, τ1, A2, τ2)
are free; τ_swap auto-correction enforces τ2 ≥ τ1.

**Audit descriptors** (single-descriptor lock deferred until cross-descriptor
robustness is verified):

| descriptor | definition |
|------------|------------|
| `tau_FG_20s` | single-exp fit on t ∈ [0.5, 20] s, V_inf shared from bi-exp |
| `tau_FG_60s` | single-exp fit on t ∈ [0.5, 60] s, V_inf shared from bi-exp |
| `t95` | first time recovery reaches 95 % toward V_inf_fit |
| `t99` | first time recovery reaches 99 % toward V_inf_fit |
| `tau_tail` | single-exp fit on t ∈ [300, 600] s, V_inf free (tail-only descriptor) |

V_inf reuse rule: `tau_FG_20s` / `tau_FG_60s` / `t95` / `t99` all use the
bi-exp `V_inf_fit` to keep the descriptor frame coherent. `tau_tail` uses
its own free V_inf because the tail-only segment is intended to probe slow
components that may differ from the bi-exp V_inf.

---

## CDEFG-compatible point extraction (audit layer)

| point | definition |
|-------|------------|
| C | nearest sample to `t_on − 0.6 s` (pre-pulse baseline) |
| D | nearest sample to `t_on + 0.02 s` |
| E | max V during pulse, excluding first 0.5 s and last 0.05 s |
| F | nearest sample to `t_off + 0.02 s` |
| G | first relaxation sample reaching 99 % recovery toward V_inf_fit |

Resistance metrics: `|ΔI| = 1.0·Q_nom_Ah`,
`R_CD = (U_D − U_C) / |ΔI|`, `R_EF = (U_E − U_F) / |ΔI|` (both expected positive
for a charge pulse). G is the point-layer counterpart of the descriptor-layer
`t99`; both are retained for audit traceability.

---

## Edge-case handling

| condition | status flag |
|-----------|-------------|
| pulse triggers V_max event | `pulse_infeasible_Vmax`, all τ NaN |
| relaxation samples insufficient | `relaxation_invalid` |
| bi-exp fit R² < 0.95 | `fit_low_quality = True`, τ still recorded |
| τ1 > τ2 returned by fit | auto-swap, `tau_swap = True` |
| τ2 hits upper bound | `tau2_bound_hit = True` |
| recovery 99 % not reached in 600 s | `recovery_99_reached = False`, t99 / G = NaN |
| individual descriptor fit fails | descriptor = NaN, set is not failed |

---

## Decision thresholds (frozen at Day 16 close)

| τ_ref distribution | criterion | next action |
|--------------------|-----------|-------------|
| **Tight** | `tau2_biexp max/min < 1.5` AND `IQR/median < 0.30`, with ≥ 3 audit descriptors agreeing | rebasing low information gain; DCAC rerun deferred in priority |
| **Broad** | `tau2_biexp max/min ≥ 2.0` OR `IQR/median ≥ 0.50`, descriptor robustness consistent | rebasing has information value; Day 18 launches rebased DCAC |
| **Intermediate** | `1.5 ≤ max/min < 2.0` OR `0.30 ≤ IQR/median < 0.50` | descriptor robustness check (cross-descriptor ranking consistency); rebasing only if `tau2_biexp` / `tau_FG_60s` / `tau_tail` rank consistently |

Even when τ_ref clusters tightly, OCV shape, voltage-headroom, and λ(Q)
differences may still drive Day 17+ outcome divergence. Tight clustering
lowers the priority of frequency rebasing but does not foreclose other
Day 17+ branches.

---

## Outputs

| file | purpose |
|------|---------|
| `data/day17_step1_tau_ref_audit.csv` | one row per set, full audit summary |
| `data/day17_step1_tau_ref_descriptor_matrix.csv` | 8 × 6 descriptor matrix |
| `data/day17_step1_relaxation_curves_long.csv.gz` | long-format relaxation V(t) curves |
| `figures/day17/...` | descriptor comparison figure (Cell 1B) |

---

## Implementation notes

Notebook 21 uses local descriptor helpers (bi-exp fit, single-exp window fits,
tail fit, recovery-time interpolation) implemented inline for audit
reproducibility. Existing project helper modules, if any, are not imported in
this first pass; modular refactoring is deferred until Day 17 close.

The DCAC main protocol grid hard constraint frozen at Day 16 close
(`|DC| + |AC| ≤ 1C`, MJ1-aligned) applies to DC–AC protocol design only.
HPPC pulse for descriptor extraction here is a diagnostic probe and is not
bound by that constraint; the 1C charge pulse is the standard HPPC amplitude
inherited from the Masterarbeit experimental pipeline.

In [1]:
# ============================================================
# Day 17 Cell 2 (final rev) — τ_ref availability audit
# 
# Pipeline:
#   1. PyBaMM Experiment with known step boundaries → ton/toff
#   2. Build DAS60-convention DataFrame with I_KL = -I_PyBaMM
#   3. Reuse helper functions from scripts/hppc_descriptor_pybamm.py
#      (no auto pulse-detection, no extract_hppc_descriptors call)
#   4. Day17-specific 600s relaxation config
#   5. Output: primary tau2 (600s window) + secondary 60s window
#              + tau_FG_eff (63.2% recovery) + t95 + t99 + tau_tail
#              + Uinf-window stability audit + D/F target error audit
# ============================================================

import sys
from pathlib import Path

PROJECT_ROOT = Path("/Users/louislu/pybamm-dcac-superimposed")
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

import pybamm
import numpy as np
import pandas as pd
import warnings
import time as _time

# Reuse algorithm-pure helpers from existing extractor (no extract_hppc_descriptors call)
from hppc_descriptor_pybamm import (
    median_in_window,
    pick_E_by_paperdef,
    pick_Uinf,
    pick_G_by_99pct,
    compute_recovery_time,
    compute_recovery_time_windowed,
    fit_descriptor_window,
    fit_tail_single_exp,
)

print("=== Day 17 Cell 2 — τ_ref availability audit (M1a path) ===")
print(f"PyBaMM: {pybamm.__version__}")
print(f"Helpers: scripts/hppc_descriptor_pybamm.py (M1a inline reuse)\n")

DATA_DIR = PROJECT_ROOT / "data"
FIG_DIR = PROJECT_ROOT / "figures" / "day17"
DATA_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

OUT_SUMMARY = DATA_DIR / "day17_step1_tau_ref_audit.csv"
OUT_DESCMAT = DATA_DIR / "day17_step1_tau_ref_descriptor_matrix.csv"
OUT_CURVES = DATA_DIR / "day17_step1_relaxation_curves_long.csv.gz"

SETS = ["Ai2020", "Chen2020", "Ecker2015", "Marquis2019",
        "Mohtat2020", "NCA_Kim2011", "OKane2022", "ORegan2022"]
CHEM_TAG = {
    "Ai2020":      "Enertech LCO/graphite",
    "Chen2020":    "LG M50 NMC811-Si",
    "Ecker2015":   "Kokam-Ecker Co-rich Mn-free",
    "Marquis2019": "Kokam-Marquis LCO/graphite",
    "Mohtat2020":  "graphite/NMC532",
    "NCA_Kim2011": "graphite/NCA",
    "OKane2022":   "LG M50 NMC811-Si",
    "ORegan2022":  "LG M50 NMC811-Si",
}

# Day17-specific config — 600s relaxation budget
DAY17_CONFIG = {
    "name": "Day17_tau_ref_audit",
    "descriptor_windows_s": [20.0, 60.0, 600.0],
    "primary_tau2_window_s": 600.0,
    "tau_fg_eff_window_s": 600.0,
    "tail_fit_start_s": 300.0,
    "tail_fit_end_s": 600.0,
    "uinf_window_start_s": 500.0,
    "uinf_window_end_s": 600.0,
}

EMPTY_DESC = {
    "tau2_biexp": np.nan,
    "tau2_secondary_60s": np.nan,
    "tau_FG_eff": np.nan,
    "t95_s": np.nan,
    "t99_s": np.nan,
    "tau_tail": np.nan,
}

PICK_HALF_WINDOW_MS = 10.0  # ±10 ms median window for CDEFG


# ============================================================
# Run audit
# ============================================================
records = []
desc_matrix = []
curve_records = []
total_t0 = _time.time()

print(f"{'Set':<14s} | {'V_init':>7s} | {'V_pre':>7s} | {'V_rest':>7s} | "
      f"{'pulse_dV':>9s} | {'relax_dV':>9s} | {'tau2_600':>8s} | {'R²':>5s}")
print("-" * 105)

for set_name in SETS:
    rec = {"param_set": set_name, "chem_tag": CHEM_TAG[set_name]}
    desc_row = {"param_set": set_name, "chem_tag": CHEM_TAG[set_name]}
    desc_row.update(EMPTY_DESC)
    t0 = _time.time()
    skip_to_exit = False

    try:
        pv = pybamm.ParameterValues(set_name)
        pv["Ambient temperature [K]"] = 293.15
        pv["Initial temperature [K]"] = 293.15
        pv.set_initial_state(0.05)

        Q_nom = float(pv["Nominal cell capacity [A.h]"])
        V_max_spec = float(pv["Upper voltage cut-off [V]"])
        V_min_spec = float(pv["Lower voltage cut-off [V]"])
        I_precharge = -0.2 * Q_nom   # PyBaMM convention I<0 = charge
        I_pulse = -1.0 * Q_nom

        rec.update({
            "Q_nom_Ah": Q_nom,
            "V_max_spec": V_max_spec, "V_min_spec": V_min_spec,
            "I_precharge_A_pybamm": I_precharge,
            "I_pulse_A_pybamm": I_pulse,
            "deltaI_A": abs(I_pulse),
        })

        rec.update({
            "descriptor_primary_window_s": DAY17_CONFIG["primary_tau2_window_s"],
            "uinf_window_start_s": DAY17_CONFIG["uinf_window_start_s"],
            "uinf_window_end_s": DAY17_CONFIG["uinf_window_end_s"],
            "tail_fit_start_s": DAY17_CONFIG["tail_fit_start_s"],
            "tail_fit_end_s": DAY17_CONFIG["tail_fit_end_s"],
        })
        
        experiment = pybamm.Experiment([
            pybamm.step.current(I_precharge, duration="900 seconds", period="5 seconds"),
            pybamm.step.rest(duration="600 seconds", period="5 seconds"),
            pybamm.step.current(I_pulse, duration="10 seconds", period="0.005 seconds"),
            pybamm.step.rest(duration="5 seconds", period="0.005 seconds"),
            pybamm.step.rest(duration="595 seconds", period="1 second"),
        ])

        model = pybamm.lithium_ion.DFN(options={"thermal": "lumped"})
        sim = pybamm.Simulation(model, parameter_values=pv, experiment=experiment)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            sol = sim.solve()

        # Flatten cycles → steps (PyBaMM 26.x cycle splitting)
        all_steps = []
        for cyc in sol.cycles:
            all_steps.extend(cyc.steps)
        if len(all_steps) < 5:
            raise RuntimeError(
                f"expected 5 steps, got {len(all_steps)}; "
                f"sol.termination={getattr(sol, 'termination', '?')}"
            )
        s_pre, s_rest, s_pulse, s_relax_a, s_relax_b = all_steps[:5]

        V_init = float(s_pre["Voltage [V]"].entries[0])
        V_after_precharge = float(s_pre["Voltage [V]"].entries[-1])
        V_after_prerest = float(s_rest["Voltage [V]"].entries[-1])

        # Build full DAS60-style trace from all 5 steps for helper-function consumption
        # I_KL = -I_PyBaMM (DAS60: I>0 = charge into battery)
        all_t_s = []
        all_V = []
        all_I_pybamm = []
        for step in all_steps[:5]:
            t_s = np.asarray(step["Time [s]"].entries, dtype=float)
            V_s = np.asarray(step["Voltage [V]"].entries, dtype=float)
            I_s = np.asarray(step["Current [A]"].entries, dtype=float)
            all_t_s.append(t_s)
            all_V.append(V_s)
            all_I_pybamm.append(I_s)
        t_s_full = np.concatenate(all_t_s)
        V_full = np.concatenate(all_V)
        I_pybamm_full = np.concatenate(all_I_pybamm)
        # Dedup at step boundaries
        keep = np.concatenate([[True], np.diff(t_s_full) > 1e-9])
        t_s_full = t_s_full[keep]
        V_full = V_full[keep]
        I_pybamm_full = I_pybamm_full[keep]

        # DAS60-convention arrays (ms time, sign-flipped current)
        t_ms_full = t_s_full * 1000.0
        I_KL_full = -I_pybamm_full

        # === Step boundaries are KNOWN — bypass detect_pulse_by_phi ===
        # Pulse step is all_steps[2]; relaxation = all_steps[3] + all_steps[4]
        t_pulse_s = np.asarray(s_pulse["Time [s]"].entries, dtype=float)
        t_on_s = float(t_pulse_s[0])
        t_off_s = float(t_pulse_s[-1])
        t_on_ms = t_on_s * 1000.0
        t_off_ms = t_off_s * 1000.0

        # I0 / Ip in DAS60 convention (charge pulse → Ip > 0)
        I0_KL = 0.0   # pre-rest is exactly 0
        Ip_KL = abs(I_pulse)  # 1C charge magnitude
        # discharge = is_discharge_pulse(I0_KL, Ip_KL) → False (charge)
        discharge = False

        # === CDEFG point picks (paper-def) ===
        delay_ms = 50.0 if discharge else 20.0
        C_target_ms = t_on_ms - 600.0
        D_target_ms = t_on_ms + delay_ms
        F_target_ms = t_off_ms + delay_ms

        Uc = median_in_window(t_ms_full, V_full, C_target_ms, PICK_HALF_WINDOW_MS / 2.0)
        Ud = median_in_window(t_ms_full, V_full, D_target_ms, PICK_HALF_WINDOW_MS / 2.0)
        Uf = median_in_window(t_ms_full, V_full, F_target_ms, PICK_HALF_WINDOW_MS / 2.0)
        E_ms = pick_E_by_paperdef(t_ms_full, V_full, t_on_ms, t_off_ms, discharge)
        Ue = median_in_window(t_ms_full, V_full, E_ms, PICK_HALF_WINDOW_MS / 2.0)

        # D/F target error (#4 audit)
        # nearest sample in raw arrays (pre-median, pre-window)
        idx_D = int(np.argmin(np.abs(t_ms_full - D_target_ms)))
        idx_F = int(np.argmin(np.abs(t_ms_full - F_target_ms)))
        D_target_error_ms = float(t_ms_full[idx_D] - D_target_ms)
        F_target_error_ms = float(t_ms_full[idx_F] - F_target_ms)

        # R0 (DAS60 convention; both R values expected positive for charge pulse)
        dI_KL = (Ip_KL - I0_KL) if abs(Ip_KL - I0_KL) > 1e-12 else 1e-12
        R0_CD = abs((Ud - Uc) / dI_KL)
        R0_EF = abs((Ue - Uf) / dI_KL)
        R0_avg = 0.5 * (R0_CD + R0_EF)

        # Pulse Vmax check
        V_pulse = np.asarray(s_pulse["Voltage [V]"].entries, dtype=float)
        pulse_max_V = float(V_pulse.max())
        pulse_hit_Vmax = pulse_max_V >= (V_max_spec - 0.005)
        V_pulse_start = float(V_pulse[0])
        V_pulse_end = float(V_pulse[-1])
        pulse_dV_mV = (V_pulse_end - V_pulse_start) * 1000.0

        # Relaxation arrays (for V_relax_end + curve persistence)
        t_relax_a = np.asarray(s_relax_a["Time [s]"].entries, dtype=float)
        V_relax_a = np.asarray(s_relax_a["Voltage [V]"].entries, dtype=float)
        t_relax_b = np.asarray(s_relax_b["Time [s]"].entries, dtype=float)
        V_relax_b = np.asarray(s_relax_b["Voltage [V]"].entries, dtype=float)
        t_relax_abs = np.concatenate([t_relax_a, t_relax_b])
        V_relax = np.concatenate([V_relax_a, V_relax_b])
        keep_r = np.concatenate([[True], np.diff(t_relax_abs) > 1e-9])
        t_relax_abs = t_relax_abs[keep_r]
        V_relax = V_relax[keep_r]
        t_relax_rel = t_relax_abs - t_relax_abs[0]
        V_relax_end = float(V_relax[-1])
        relax_dV_mV = (V_relax_end - V_relax[0]) * 1000.0

        rec.update({
            "V_init_charge": V_init,
            "V_after_precharge": V_after_precharge,
            "V_after_prerest": V_after_prerest,
            "V_pulse_start": V_pulse_start,
            "V_pulse_max": pulse_max_V,
            "V_pulse_end": V_pulse_end,
            "pulse_dV_mV": pulse_dV_mV,
            "V_relax_end": V_relax_end,
            "relax_dV_mV": relax_dV_mV,
            "pulse_hit_Vmax": pulse_hit_Vmax,
            "ton_ms": t_on_ms, "toff_ms": t_off_ms,
            "C_ms": C_target_ms, "D_target_ms": D_target_ms,
            "E_ms": E_ms, "F_target_ms": F_target_ms,
            "Uc_V": Uc, "Ud_V": Ud, "Ue_V": Ue, "Uf_V": Uf,
            "D_target_error_ms": D_target_error_ms,
            "F_target_error_ms": F_target_error_ms,
            "I0_KL_DAS60_A": I0_KL, "Ip_KL_DAS60_A": Ip_KL,
            "R0_CD_Ohm": R0_CD, "R0_EF_Ohm": R0_EF, "R0_avg_Ohm": R0_avg,
        })

        if pulse_hit_Vmax:
            rec["status"] = "pulse_infeasible_Vmax"
            for k in ["Uinf_V", "Uinf_window_start_V", "Uinf_window_end_V",
                     "Uinf_window_dV_mV", "Uinf_window_abs_dV_mV",
                     "Uinf_window_class",
                     "tau1_biexp", "tau2_biexp", "primary_R2", "primary_RMSE",
                     "tau1_secondary_60s", "tau2_secondary_60s", "secondary_R2",
                     "tau_FG_eff", "t95_s", "t99_s", "tau_tail", "tail_R2",
                     "G_ms", "Ug_V"]:
                rec[k] = np.nan
            rec.update({
                "biexp_fit_status": "skipped_Vmax",
                "tail_fit_status": "skipped_Vmax",
                "fit_low_quality": False,
                "recovery_99_reached": False,
            })
            print(f"{set_name:<14s} | {V_init:>7.3f} | {V_after_precharge:>7.3f} | "
                  f"{V_after_prerest:>7.3f} | {pulse_dV_mV:>+9.1f} | "
                  f"{relax_dV_mV:>+9.1f} | {'-':>8s} | {'-':>5s}  pulse_hit_Vmax")
            skip_to_exit = True

        if not skip_to_exit:
            # === Uinf via cell_config window [t_off + 500s, t_off + 600s] ===
            Uinf = pick_Uinf(t_ms_full, V_full, t_off_ms, DAY17_CONFIG)
            # Uinf-window stability audit
            uinf_start_target_ms = t_off_ms + DAY17_CONFIG["uinf_window_start_s"] * 1000.0
            uinf_end_target_ms = t_off_ms + DAY17_CONFIG["uinf_window_end_s"] * 1000.0
            idx_uinf_start = int(np.argmin(np.abs(t_ms_full - uinf_start_target_ms)))
            idx_uinf_end = int(np.argmin(np.abs(t_ms_full - uinf_end_target_ms)))
            Uinf_window_start_V = float(V_full[idx_uinf_start])
            Uinf_window_end_V = float(V_full[idx_uinf_end])
            Uinf_window_dV_mV = (Uinf_window_end_V - Uinf_window_start_V) * 1000.0
            Uinf_window_abs_dV_mV = abs(Uinf_window_dV_mV)
            if Uinf_window_abs_dV_mV < 1.0:
                Uinf_window_class = "Uinf_window_stable"
            elif Uinf_window_abs_dV_mV < 5.0:
                Uinf_window_class = "Uinf_window_marginal"
            else:
                Uinf_window_class = "Uinf_window_unstable"

            rec.update({
                "Uinf_V": Uinf,
                "Uinf_window_start_V": Uinf_window_start_V,
                "Uinf_window_end_V": Uinf_window_end_V,
                "Uinf_window_dV_mV": Uinf_window_dV_mV,
                "Uinf_window_abs_dV_mV": Uinf_window_abs_dV_mV,
                "Uinf_window_class": Uinf_window_class,
            })

            # === Multi-window bi-exp fits (primary 600s + secondary 60s) ===
            primary_ws = DAY17_CONFIG["primary_tau2_window_s"]
            fit_primary = fit_descriptor_window(t_ms_full, V_full, F_target_ms, primary_ws)
            fit_secondary = fit_descriptor_window(t_ms_full, V_full, F_target_ms, 60.0)

            rec.update({
                "tau1_biexp": fit_primary.tau1,
                "tau2_biexp": fit_primary.tau2,
                "primary_R2": fit_primary.r2,
                "primary_RMSE": fit_primary.rmse,
                "primary_window_s": primary_ws,
                "biexp_fit_status": "ok" if fit_primary.success else "failed",
                "fit_low_quality": ((fit_primary.r2 < 0.95)
                                    if fit_primary.success else True),
                "tau1_secondary_60s": fit_secondary.tau1,
                "tau2_secondary_60s": fit_secondary.tau2,
                "secondary_R2": fit_secondary.r2,
            })

            # === Recovery descriptors using Uinf_window (NOT bi-exp V_inf) ===
            # tau_FG_eff: 63.2% recovery time (windowed to primary 600s)
            tau_FG_eff = compute_recovery_time_windowed(
                t_ms_full, V_full, F_target_ms, Uf, Uinf,
                0.632, DAY17_CONFIG["tau_fg_eff_window_s"]
            )
            t95 = compute_recovery_time(t_ms_full, V_full, F_target_ms, Uf, Uinf, 0.95)
            t99 = compute_recovery_time(t_ms_full, V_full, F_target_ms, Uf, Uinf, 0.99)
            recov_ok = not np.isnan(t99)
            rec.update({
                "tau_FG_eff": tau_FG_eff,
                "t95_s": t95,
                "t99_s": t99,
                "recovery_99_reached": recov_ok,
            })

            # === G point (linked to Uinf-based 99% recovery) ===
            G_ms, Ug = pick_G_by_99pct(t_ms_full, V_full, t_off_ms, Uf, Uinf)
            rec["G_ms"] = G_ms
            rec["Ug_V"] = Ug

            # === Tail-only single-exp fit ===
            tail_start_ms = F_target_ms + DAY17_CONFIG["tail_fit_start_s"] * 1000.0
            tail_end_ms = F_target_ms + DAY17_CONFIG["tail_fit_end_s"] * 1000.0
            tail_mask = (t_ms_full >= tail_start_ms) & (t_ms_full <= tail_end_ms)
            tau_tail = np.nan
            tail_R2 = np.nan
            tail_status = "insufficient_samples"
            if int(tail_mask.sum()) >= 10:
                try:
                    t_tail_s = (t_ms_full[tail_mask] - tail_start_ms) / 1000.0
                    V_tail = V_full[tail_mask]
                    popt_t, rmse_t, r2_t = fit_tail_single_exp(t_tail_s, V_tail)
                    tau_tail = float(popt_t[2])
                    tail_R2 = float(r2_t)
                    tail_status = "ok"
                except Exception as e:
                    tail_status = f"tailfit_fail:{type(e).__name__}"
            rec.update({
                "tau_tail": tau_tail,
                "tail_R2": tail_R2,
                "tail_fit_status": tail_status,
            })

            rec["status"] = "ok"

            # Long-format curve points (relaxation only, t_rel measured from F point)
            t_F_s = F_target_ms / 1000.0
            for t_abs_s, V_v in zip(t_relax_abs, V_relax):
                curve_records.append({
                    "param_set": set_name,
                    "chem_tag": CHEM_TAG[set_name],
                    "t_relax_s": float(t_abs_s - t_F_s),
                    "V_V": float(V_v),
                })

            # Descriptor matrix update
            desc_row.update({
                "tau2_biexp": rec["tau2_biexp"],
                "tau2_secondary_60s": rec["tau2_secondary_60s"],
                "tau_FG_eff": rec["tau_FG_eff"],
                "t95_s": rec["t95_s"],
                "t99_s": rec["t99_s"],
                "tau_tail": rec["tau_tail"],
            })

            tau2_str = (f"{rec['tau2_biexp']:>8.2f}"
                        if not np.isnan(rec['tau2_biexp']) else f"{'NaN':>8s}")
            r2_str = (f"{rec['primary_R2']:>5.3f}"
                      if not np.isnan(rec['primary_R2']) else f"{'NaN':>5s}")
            uinf_flag = (
                f" {Uinf_window_class.replace('Uinf_window_', '')}"
                if Uinf_window_class != "Uinf_window_stable" else ""
            )
            print(f"{set_name:<14s} | {V_init:>7.3f} | {V_after_precharge:>7.3f} | "
                  f"{V_after_prerest:>7.3f} | {pulse_dV_mV:>+9.1f} | "
                  f"{relax_dV_mV:>+9.1f} | {tau2_str} | {r2_str}{uinf_flag}")

    except Exception as e:
        rec["status"] = f"solve_error: {type(e).__name__}"
        rec["error_msg"] = str(e)[:200]
        for k in ["Q_nom_Ah", "V_init_charge", "V_after_precharge", "V_after_prerest",
                  "tau1_biexp", "tau2_biexp", "primary_R2",
                  "tau2_secondary_60s", "tau_FG_eff", "t95_s", "t99_s", "tau_tail"]:
            rec.setdefault(k, np.nan)
        print(f"{set_name:<14s} | FAIL  {type(e).__name__}: {str(e)[:80]}")

    rec["wall_time_s"] = round(_time.time() - t0, 2)
    records.append(rec)
    desc_matrix.append(desc_row)


# ============================================================
# Save outputs
# ============================================================
df_summary = pd.DataFrame(records)
df_summary.to_csv(OUT_SUMMARY, index=False)
df_desc = pd.DataFrame(desc_matrix)
df_desc.to_csv(OUT_DESCMAT, index=False)
df_curves = pd.DataFrame(curve_records)
df_curves.to_csv(OUT_CURVES, index=False, compression="gzip")

# ============================================================
# CDEFG D/F point fidelity audit
# ============================================================
print("\n" + "=" * 100)
print("CDEFG point fidelity audit (D/F target = 20 ms after pulse on/off)")
print("=" * 100)
fid_cols = ["param_set", "D_target_error_ms", "F_target_error_ms"]
ok_mask = ~df_summary["status"].fillna("").str.startswith("solve_error")
fid_df = df_summary[ok_mask][fid_cols]
print(fid_df.to_string(index=False, float_format=lambda x: f"{x:>+8.3f}"))
if len(fid_df):
    max_D = fid_df["D_target_error_ms"].abs().max()
    max_F = fid_df["F_target_error_ms"].abs().max()
    print(f"\nMax |D_target_error| = {max_D:.3f} ms  "
          f"({'OK' if max_D < 5.0 else 'WARN: >5ms'})")
    print(f"Max |F_target_error| = {max_F:.3f} ms  "
          f"({'OK' if max_F < 5.0 else 'WARN: >5ms'})")

# ============================================================
# Uinf-window stability audit
# ============================================================
print("\n" + "=" * 100)
print("Uinf window stability audit (V drift over [t_off+500s, t_off+600s])")
print("=" * 100)
if "Uinf_window_dV_mV" in df_summary.columns:
    uinf_cols = ["param_set", "Uinf_window_start_V", "Uinf_window_end_V",
                 "Uinf_window_dV_mV", "Uinf_window_class"]
    print(df_summary[ok_mask][uinf_cols].to_string(index=False))

# ============================================================
# Descriptor matrix
# ============================================================
print("\n" + "=" * 100)
print("Day 17 — descriptor matrix (8 sets × 6 descriptors)")
print("=" * 100)
desc_cols = ["tau2_biexp", "tau2_secondary_60s", "tau_FG_eff",
             "t95_s", "t99_s", "tau_tail"]
print(df_desc[["param_set"] + desc_cols].to_string(
    index=False,
    float_format=lambda x: f"{x:>8.2f}" if not np.isnan(x) else "     NaN"))

print("\nDescriptor distribution stats:")
for col in desc_cols:
    vals = df_desc[col].dropna()
    if len(vals) >= 2:
        med = float(vals.median())
        iqr = float(vals.quantile(0.75) - vals.quantile(0.25))
        rng = float(vals.max() / vals.min()) if vals.min() > 0 else np.nan
        iqr_med = iqr / med if med > 0 else np.nan
        print(f"  {col:<22s}: n={len(vals)}, median={med:.2f}s, "
              f"IQR/median={iqr_med:.3f}, max/min={rng:.3f}")
    else:
        print(f"  {col:<22s}: n={len(vals)} (insufficient)")

# ============================================================
# Decision threshold (frozen at Day 16 close)
# ============================================================
print("\n" + "=" * 100)
print("τ_ref availability decision (frozen thresholds, primary = tau2_biexp 600s)")
print("=" * 100)
tau2 = df_desc["tau2_biexp"].dropna()
if len(tau2) >= 2:
    rng = float(tau2.max() / tau2.min()) if tau2.min() > 0 else np.nan
    iqr_med = float((tau2.quantile(0.75) - tau2.quantile(0.25)) / tau2.median())
    print(f"  tau2_biexp max/min   = {rng:.3f}  "
          f"(<1.5 tight, ≥2.0 broad, intermediate otherwise)")
    print(f"  tau2_biexp IQR/median = {iqr_med:.3f}  "
          f"(<0.30 tight, ≥0.50 broad, intermediate otherwise)")
    if rng < 1.5 and iqr_med < 0.30:
        verdict = ("TIGHT — frequency rebasing low information gain; "
                   "DCAC rerun deferred in priority")
    elif rng >= 2.0 or iqr_med >= 0.50:
        verdict = ("BROAD — frequency rebasing has information value; "
                   "DCAC rerun justified")
    else:
        verdict = "INTERMEDIATE — descriptor robustness check needed"
    print(f"  Verdict: {verdict}")
else:
    print(f"  Insufficient successful tau2_biexp fits "
          f"(n={len(tau2)}); verdict deferred")

print(f"\nWall time: {round(_time.time() - total_t0, 1)}s")
print(f"\n[wrote] {OUT_SUMMARY}")
print(f"[wrote] {OUT_DESCMAT}")
print(f"[wrote] {OUT_CURVES}  ({len(df_curves)} points)")

=== Day 17 Cell 2 — τ_ref availability audit (M1a path) ===
PyBaMM: 26.3.1
Helpers: scripts/hppc_descriptor_pybamm.py (M1a inline reuse)

Set            |  V_init |   V_pre |  V_rest |  pulse_dV |  relax_dV | tau2_600 |    R²
---------------------------------------------------------------------------------------------------------
Ai2020         |   3.586 |   3.676 |   3.640 |      +2.7 |      -4.1 |    52.42 | 1.000
Chen2020       |   3.166 |   3.368 |   3.292 |     +52.0 |     -43.1 |    36.32 | 1.000
Ecker2015      |   3.311 |   3.470 |   3.450 |     +15.9 |     -12.6 |    16.52 | 1.000
Marquis2019    |   3.662 |   3.721 |   3.676 |      +5.5 |      -6.5 |    43.45 | 1.000
Mohtat2020     |   3.498 |   3.541 |   3.520 |      +6.4 |      -6.8 |    43.93 | 1.000
NCA_Kim2011    |   3.197 |   3.248 |   3.240 |     +11.5 |      -8.8 |    39.39 | 1.000
OKane2022      |   3.186 |   3.387 |   3.313 |     +48.4 |     -47.0 |    58.82 | 1.000
ORegan2022     |   3.109 |   3.341 |   3.263 |     +

In [3]:
# ============================================================
# Day 17 Cell 3 — relaxation-shape + descriptor-robustness visualization
# 
# Pure visualization cell. Reads Cell 2 outputs only, no PyBaMM rerun.
# 
# Inputs:
#   data/day17_step1_tau_ref_audit.csv
#   data/day17_step1_tau_ref_descriptor_matrix.csv
#   data/day17_step1_relaxation_curves_long.csv.gz
# 
# Outputs:
#   figures/day17/C1_relaxation_voltage_overlay.png
#   figures/day17/C2_normalized_recovery_overlay.png
#   figures/day17/C3_semilog_remaining_overlay.png
#   figures/day17/C4_descriptor_rank_heatmap.png
# 
# Purpose:
#   Validate whether tau2_biexp BROAD verdict is supported by genuine
#   relaxation-shape time-scale spread (Figures C2/C3) or driven by
#   fit-window / amplitude / descriptor artifacts (Figure C4 cross-
#   descriptor ranking).
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from pathlib import Path

DATA_DIR = Path("/Users/louislu/pybamm-dcac-superimposed") / "data"
FIG_DIR = Path("/Users/louislu/pybamm-dcac-superimposed") / "figures" / "day17"
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("=== Day 17 Cell 3 — relaxation-shape + descriptor-robustness viz ===\n")

df_summary = pd.read_csv(DATA_DIR / "day17_step1_tau_ref_audit.csv")
df_desc = pd.read_csv(DATA_DIR / "day17_step1_tau_ref_descriptor_matrix.csv")
df_curves = pd.read_csv(DATA_DIR / "day17_step1_relaxation_curves_long.csv.gz")

print(f"Summary rows: {len(df_summary)}")
print(f"Descriptor matrix rows: {len(df_desc)}")
print(f"Curve points: {len(df_curves)}\n")

# Set order grouped by chemistry family (consistent with Day 16 Cell 3B)
SET_ORDER = [
    "Ai2020", "Marquis2019",                  # LCO
    "Chen2020", "OKane2022", "ORegan2022",    # LG M50 lineage
    "Mohtat2020",                             # NMC532
    "NCA_Kim2011",                            # NCA
    "Ecker2015",                              # Kokam-Ecker
]
CHEM_FAMILY = {
    "Ai2020": "LCO", "Marquis2019": "LCO",
    "Chen2020": "LG M50", "OKane2022": "LG M50", "ORegan2022": "LG M50",
    "Mohtat2020": "NMC532",
    "NCA_Kim2011": "NCA",
    "Ecker2015": "Kokam-Ecker",
}
FAMILY_COLOR = {
    "LCO":         "#d62728",
    "LG M50":      "#1f77b4",
    "NMC532":      "#2ca02c",
    "NCA":         "#9467bd",
    "Kokam-Ecker": "#ff7f0e",
}
SET_MARKER = {
    "Ai2020": "o",   "Marquis2019": "s",
    "Chen2020": "o", "OKane2022": "^", "ORegan2022": "v",
    "Mohtat2020": "o",
    "NCA_Kim2011": "o",
    "Ecker2015": "o",
}
# Linestyle to disambiguate LG M50 lineage when overlaid
SET_LINESTYLE = {
    "Ai2020": "-",   "Marquis2019": "--",
    "Chen2020": "-", "OKane2022": "--", "ORegan2022": ":",
    "Mohtat2020": "-",
    "NCA_Kim2011": "-",
    "Ecker2015": "-",
}


def get_set_curve_from_F(set_name):
    """Return (t_from_F_s, V_V) arrays for a given set.
    
    Cell 2 long-format CSV stores t_relax_s as time-after-relaxation-step-start
    (= time after toff). F point is toff + 20 ms by CDEFG paper-def.
    This helper shifts the time axis so t = 0 corresponds to F, ensuring all
    Cell 3 figure x-axes are F-based and the t >= 0 mask truly excludes the
    pre-F transition segment.
    """
    sub = df_curves[df_curves["param_set"] == set_name].sort_values("t_relax_s")
    t_relax = sub["t_relax_s"].to_numpy()
    V = sub["V_V"].to_numpy()

    row = df_summary[df_summary["param_set"] == set_name].iloc[0]
    # F_target_ms = toff_ms + 20 ms (charge pulse) by paper-def;
    # relaxation curve t_relax_s starts at toff. Shift to F-based.
    F_offset_s = (float(row["F_target_ms"]) - float(row["toff_ms"])) / 1000.0

    t_from_F = t_relax - F_offset_s
    return t_from_F, V


def get_set_Uinf(set_name):
    """Return Uinf_V from summary table."""
    row = df_summary[df_summary["param_set"] == set_name].iloc[0]
    return float(row["Uinf_V"])


def get_set_UF(set_name):
    """Return Uf_V from summary table (V at F point)."""
    row = df_summary[df_summary["param_set"] == set_name].iloc[0]
    return float(row["Uf_V"])


# ============================================================
# Figure C1 — raw relaxation voltage overlay
# ============================================================
print("Generating C1 (raw voltage overlay) ...")
fig, ax = plt.subplots(figsize=(10, 6))

for s in SET_ORDER:
    t, V = get_set_curve_from_F(s)
    # Filter t >= 0 (after F point)
    mask = t >= 0
    fam = CHEM_FAMILY[s]
    ax.plot(t[mask], V[mask],
            color=FAMILY_COLOR[fam],
            linestyle=SET_LINESTYLE[s],
            linewidth=1.6,
            alpha=0.85,
            label=f"{s} [{fam}]")

ax.set_xlabel("Time after F point  [s]", fontsize=11)
ax.set_ylabel("Voltage  [V]", fontsize=11)
ax.set_title("C1: Raw relaxation voltage overlay (8 sets, post-F)",
             fontsize=11)
ax.legend(loc="best", fontsize=8, framealpha=0.9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
out_c1 = FIG_DIR / "C1_relaxation_voltage_overlay.png"
fig.savefig(out_c1, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"  [wrote] {out_c1}")


# ============================================================
# Figure C2 — normalized recovery overlay
#   recovery(t) = 1 - |V(t) - Uinf| / |UF - Uinf|
# ============================================================
print("Generating C2 (normalized recovery overlay) ...")
fig, ax = plt.subplots(figsize=(10, 6))

for s in SET_ORDER:
    t, V = get_set_curve_from_F(s)
    Uinf = get_set_Uinf(s)
    UF = get_set_UF(s)
    denom = abs(UF - Uinf)
    if denom < 1e-9:
        continue
    remaining = np.abs(V - Uinf) / denom
    recovery = 1.0 - remaining
    mask = t >= 0
    fam = CHEM_FAMILY[s]
    ax.plot(t[mask], recovery[mask],
            color=FAMILY_COLOR[fam],
            linestyle=SET_LINESTYLE[s],
            linewidth=1.6,
            alpha=0.85,
            label=f"{s} [{fam}]")

ax.axhline(0.632, color="black", linewidth=0.6, linestyle=":", alpha=0.5)
ax.text(0.99, 0.632, " 63.2% (τ_FG_eff)",
        transform=ax.get_yaxis_transform(),
        fontsize=8, va="center", color="gray")
ax.axhline(0.95, color="black", linewidth=0.6, linestyle=":", alpha=0.5)
ax.text(0.99, 0.95, " 95% (t95)",
        transform=ax.get_yaxis_transform(),
        fontsize=8, va="center", color="gray")
ax.axhline(0.99, color="black", linewidth=0.6, linestyle=":", alpha=0.5)
ax.text(0.99, 0.99, " 99% (t99)",
        transform=ax.get_yaxis_transform(),
        fontsize=8, va="center", color="gray")

ax.set_xlabel("Time after F point  [s]", fontsize=11)
ax.set_ylabel(r"Normalized recovery $1 - |V(t)-U_\infty|/|U_F-U_\infty|$",
              fontsize=11)
ax.set_title("C2: Normalized recovery overlay (amplitude removed)\n"
             "Curves spread → genuine time-scale difference; "
             "curves collapse → BROAD is amplitude/fit-window artifact",
             fontsize=10)
ax.set_ylim(-0.05, 1.05)
ax.legend(loc="lower right", fontsize=8, framealpha=0.9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
out_c2 = FIG_DIR / "C2_normalized_recovery_overlay.png"
fig.savefig(out_c2, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"  [wrote] {out_c2}")


# ============================================================
# Figure C3 — semilog remaining fraction (x ∈ [0, 300]s)
# ============================================================
print("Generating C3 (semilog remaining) ...")
fig, ax = plt.subplots(figsize=(10, 6))

for s in SET_ORDER:
    t, V = get_set_curve_from_F(s)
    Uinf = get_set_Uinf(s)
    UF = get_set_UF(s)
    denom = abs(UF - Uinf)
    if denom < 1e-9:
        continue
    remaining = np.abs(V - Uinf) / denom
    # Clip remaining to avoid log(0); also restrict x to [0, 300]s
    mask = (t >= 0) & (t <= 300.0) & (remaining > 1e-4)
    fam = CHEM_FAMILY[s]
    ax.plot(t[mask], remaining[mask],
            color=FAMILY_COLOR[fam],
            linestyle=SET_LINESTYLE[s],
            linewidth=1.6,
            alpha=0.85,
            label=f"{s} [{fam}]")

ax.set_yscale("log")
ax.set_xlabel("Time after F point  [s]  (window 0–300 s)", fontsize=11)
ax.set_ylabel(r"Remaining fraction $|V(t)-U_\infty|/|U_F-U_\infty|$  (log)",
              fontsize=11)
ax.set_title("C3: Semilog remaining-fraction overlay\n"
             "Straight line → single time constant; "
             "curvature → multi-τ; spread → cross-set time-scale difference",
             fontsize=10)
ax.legend(loc="upper right", fontsize=8, framealpha=0.9)
ax.grid(True, alpha=0.3, which="both")
plt.tight_layout()
out_c3 = FIG_DIR / "C3_semilog_remaining_overlay.png"
fig.savefig(out_c3, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"  [wrote] {out_c3}")


# ============================================================
# Figure C4 — descriptor rank heatmap
# ============================================================
print("Generating C4 (descriptor rank heatmap) ...")
desc_cols = ["tau2_biexp", "tau2_secondary_60s", "tau_FG_eff",
             "t95_s", "t99_s", "tau_tail"]
desc_labels = ["τ2_biexp\n(600s win)", "τ2_secondary\n(60s win)",
               "τ_FG_eff\n(63.2%)", "t95", "t99", "τ_tail"]

# Build matrix: rows = SET_ORDER, cols = descriptors
# Each column independently ranked 1=smallest to 8=largest
df_desc_indexed = df_desc.set_index("param_set")
n_sets = len(SET_ORDER)
n_desc = len(desc_cols)
mat = np.full((n_sets, n_desc), np.nan)
val_mat = np.full((n_sets, n_desc), np.nan)
for j, col in enumerate(desc_cols):
    vals = df_desc_indexed.loc[SET_ORDER, col].to_numpy()
    val_mat[:, j] = vals
    # Rank ascending: smallest = 1
    ranks = pd.Series(vals).rank(method="average").to_numpy()
    mat[:, j] = ranks

fig, ax = plt.subplots(figsize=(8.5, 6))
cmap = plt.get_cmap("RdYlBu_r")  # red high rank, blue low rank
im = ax.imshow(mat, cmap=cmap, aspect="auto", vmin=1, vmax=n_sets)

ax.set_xticks(range(n_desc))
ax.set_xticklabels(desc_labels, fontsize=9)
ax.set_yticks(range(n_sets))
ax.set_yticklabels([f"{s}\n[{CHEM_FAMILY[s]}]" for s in SET_ORDER], fontsize=8)

# Annotate each cell with rank + raw value
for i in range(n_sets):
    for j in range(n_desc):
        rank = mat[i, j]
        val = val_mat[i, j]
        if np.isnan(rank):
            txt = "—"
        else:
            txt = f"#{int(round(rank))}\n{val:.1f}s"
        # Text color: white on dark, black on light
        color = "white" if (rank >= 6 or rank <= 2) else "black"
        ax.text(j, i, txt, ha="center", va="center",
                fontsize=7.5, color=color)

cbar = plt.colorbar(im, ax=ax, shrink=0.85)
cbar.set_label("Rank (1 = smallest descriptor value, 8 = largest)",
               fontsize=10)
ax.set_title("C4: Descriptor rank heatmap (independent ranking per descriptor)\n"
             "Vertical color stripes → consistent ranking across descriptors;\n"
             "Mixed colors per row → cross-descriptor inconsistency (Caveat 2)",
             fontsize=10)
plt.tight_layout()
out_c4 = FIG_DIR / "C4_descriptor_rank_heatmap.png"
fig.savefig(out_c4, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"  [wrote] {out_c4}")


# ============================================================
# Cross-descriptor ranking quantification (textual audit)
# ============================================================
print("\n" + "=" * 80)
print("Cross-descriptor ranking robustness (Spearman correlation matrix)")
print("=" * 80)
rank_df = pd.DataFrame(mat, index=SET_ORDER, columns=desc_cols)
corr = rank_df.corr(method="spearman")
print(corr.round(3).to_string())

# Concordance: average pairwise Spearman correlation across descriptor pairs
n_pairs = 0
sum_corr = 0.0
for i in range(n_desc):
    for j in range(i + 1, n_desc):
        sum_corr += corr.iloc[i, j]
        n_pairs += 1
mean_pairwise = sum_corr / n_pairs if n_pairs > 0 else np.nan

print(f"\nMean pairwise Spearman correlation across all descriptor pairs: {mean_pairwise:.3f}")
if mean_pairwise >= 0.85:
    rank_consistency = "HIGH consistency — descriptors rank sets similarly"
elif mean_pairwise >= 0.60:
    rank_consistency = "MODERATE consistency — partial agreement"
else:
    rank_consistency = "LOW consistency — descriptor choice strongly affects ranking"
print(f"Interpretation: {rank_consistency}")

# ============================================================
# Summary
# ============================================================
print("\n" + "=" * 80)
print("Cell 3 outputs")
print("=" * 80)
for f in sorted(FIG_DIR.glob("C*.png")):
    print(f"  {f.name}  ({f.stat().st_size / 1024:.0f} KB)")

print("\nVisual inspection checklist:")
print("  C1 — raw shape difference; LG M50 lineage three lines should be close")
print("  C2 — KEY: normalized curves spread (BROAD genuine) or collapse (BROAD artifact)?")
print("  C3 — straight or curved log-lines; multi-τ visible?")
print("  C4 — vertical color stripes (consistent rank) or mixed (Caveat 2 confirmed)?")
print(f"  Spearman mean: {mean_pairwise:.3f}  ({rank_consistency})")

=== Day 17 Cell 3 — relaxation-shape + descriptor-robustness viz ===

Summary rows: 8
Descriptor matrix rows: 8
Curve points: 12768

Generating C1 (raw voltage overlay) ...
  [wrote] /Users/louislu/pybamm-dcac-superimposed/figures/day17/C1_relaxation_voltage_overlay.png
Generating C2 (normalized recovery overlay) ...
  [wrote] /Users/louislu/pybamm-dcac-superimposed/figures/day17/C2_normalized_recovery_overlay.png
Generating C3 (semilog remaining) ...
  [wrote] /Users/louislu/pybamm-dcac-superimposed/figures/day17/C3_semilog_remaining_overlay.png
Generating C4 (descriptor rank heatmap) ...
  [wrote] /Users/louislu/pybamm-dcac-superimposed/figures/day17/C4_descriptor_rank_heatmap.png

Cross-descriptor ranking robustness (Spearman correlation matrix)
                    tau2_biexp  tau2_secondary_60s  tau_FG_eff  t95_s  t99_s  tau_tail
tau2_biexp               1.000               0.476       0.303  0.571  0.667     0.310
tau2_secondary_60s       0.476               1.000       0.691  0.5